# Cleanup — Tear Down Demo Resources

Removes all resources created by the demo that `databricks bundle destroy` cannot handle:
- Synced tables
- Lakebase project
- Schemas and tables

**Run `databricks bundle destroy` AFTER this notebook to remove jobs, pipeline, and app.**

In [ ]:
%pip install -U "databricks-sdk>=0.81.0"
dbutils.library.restartPython()

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

CATALOG = "startups_catalog"
PROJECT_ID = "db-residential-copilot"

## 1. Delete Synced Tables

In [ ]:
synced_tables = [
    f"{CATALOG}.dbx_res_gold.portfolio_metrics",
    f"{CATALOG}.dbx_res_gold.portfolio_time_series",
]

for synced_table_id in synced_tables:
    try:
        w.api_client.do(
            "DELETE",
            f"/api/2.0/postgres/synced_tables/{synced_table_id}",
        )
        print(f"Deleted synced table: {synced_table_id}")
    except Exception as e:
        print(f"Skip {synced_table_id}: {e}")

## 2. Delete Lakebase Project

In [ ]:
try:
    w.postgres.delete_project(name=f"projects/{PROJECT_ID}")
    print(f"Deleted Lakebase project: {PROJECT_ID}")
except Exception as e:
    print(f"Skip Lakebase project: {e}")

## 3. Drop Schemas (dbx_res_gold, dbx_res_silver, dbx_res_bronze, dbx_res_raw)

**Caution:** This drops all tables in these schemas. Only run if you want a full teardown.

In [ ]:
schemas_to_drop = ["dbx_res_gold", "dbx_res_silver", "dbx_res_bronze", "dbx_res_raw"]

for schema in schemas_to_drop:
    try:
        spark.sql(f"DROP SCHEMA IF EXISTS {CATALOG}.{schema} CASCADE")
        print(f"Dropped schema: {CATALOG}.{schema}")
    except Exception as e:
        print(f"Skip schema {schema}: {e}")

print("\nCleanup complete. Now run: databricks bundle destroy -t dev -p vm")